# SecureFinger — improved FVC fine-tuning (~30 min)
Runtime → Change runtime type → **T4 GPU**. Results are saved to Google Drive (`MyDrive/securefinger_results_fvc_v2`), so a disconnect loses nothing. On the Mac run `caffeinate -d` in Terminal so the screen doesn't sleep.

## 1. Upload the NEW `securefinger.zip` and install

In [ ]:
from google.colab import files
files.upload()  # choose securefinger.zip
!unzip -q -o securefinger.zip
%cd /content/securefinger
!pip -q install tenseal fastapi httpx cryptography opencv-python-headless pyjwt scikit-learn

## 2. Mount Drive and point to the SOCOFing encoder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/securefinger_results_fvc_v2'
WEIGHTS = '/content/drive/MyDrive/securefinger_results_ext/encoder_seed0.pt'
!mkdir -p "$OUT"
!ls -la "$WEIGHTS"
!nvidia-smi --query-gpu=name --format=csv

## 3. Main run: 192 px input + stronger augmentation (~20–25 min)
Downloads FVC2000/2002/2004 set B + Neurotechnology again (a minute or two), then 3 seeds × 2 folds of fine-tuning. Check the database list shows 8 impressions per finger.

In [ ]:
!python experiments/fvc_v2.py --root /content/fvc --weights "$WEIGHTS" --size 192 --epochs 60 --out "$OUT/size192" 2>&1 | tee "$OUT/size192_log.txt"

## 4. Optional, only if time is left (~10 min): same recipe at 96 px
This tells us whether the gain comes from the resolution or from the training recipe. Skip it if you're short on time.

In [ ]:
!python experiments/fvc_v2.py --root /content/fvc --no-download --weights "$WEIGHTS" --size 96 --epochs 60 --out "$OUT/size96" 2>&1 | tee "$OUT/size96_log.txt"

## 5. Summary to paste back

In [ ]:
import json, os
for name in ['size192', 'size96']:
    p = f'{OUT}/{name}/fvc_v2.json'
    if not os.path.exists(p):
        print(name, ': not run'); continue
    r = json.load(open(p))
    print('==', name, r['config'], f"{r.get('minutes', 0):.1f} min")
    for k, v in r.get('summary', {}).items():
        print(f'   {k:18s} {v["mean"]*100:6.2f} ± {v["std"]*100:.2f} %')
    for d, v in r.get('per_db_eer', {}).items():
        print(f'   {d:16s} EER {v["mean"]*100:6.2f} ± {v["std"]*100:.2f} %')

## 6. Download results

In [ ]:
!cd "$OUT" && zip -q -r /content/results_fvc_v2.zip .
from google.colab import files
files.download('/content/results_fvc_v2.zip')